In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
bronze_table = "observatorio_dev.bronze.plantas_reservorios"
silver_table = "observatorio_dev.silver.plantas_reservorios"

In [0]:
bronze_df = spark.table(bronze_table)

bronze_count = bronze_df.count()

assert bronze_count > 0, "La tabla bronze.plantas_reservorios está vacía."

print("Registros en Bronze plantas_reservorios:", bronze_count)

In [0]:
expected_columns = [
    "id",
    "region",
    "nombre_planta",
    "nombre_reservorio",
    "source_file_name",
    "source_file_path",
    "ingestion_timestamp",
    "load_date"
]

missing_columns = [
    column_name for column_name in expected_columns
    if column_name not in bronze_df.columns
]

assert not missing_columns, f"Faltan columnas en Bronze: {missing_columns}"

print("Validación OK: Bronze tiene las columnas esperadas.")

In [0]:
silver_df = (
    bronze_df
    .select(
        F.col("id"),
        F.upper(F.trim(F.col("region"))).alias("region"),
        F.upper(F.trim(F.col("nombre_planta"))).alias("nombre_planta"),
        F.upper(F.trim(F.col("nombre_reservorio"))).alias("nombre_reservorio"),
        F.col("source_file_name"),
        F.col("source_file_path"),
        F.col("ingestion_timestamp"),
        F.col("load_date"),
        F.current_timestamp().alias("silver_created_at"),
        F.current_timestamp().alias("silver_updated_at")
    )
)

In [0]:
duplicates_df = (
    silver_df
    .groupBy("nombre_planta", "nombre_reservorio")
    .count()
    .filter(F.col("count") > 1)
)

duplicates_count = duplicates_df.count()

print("Relaciones duplicadas antes de deduplicar:", duplicates_count)

display(duplicates_df)

In [0]:
window_spec = (
    Window
    .partitionBy("nombre_planta", "nombre_reservorio")
    .orderBy(
        F.col("load_date").desc_nulls_last(),
        F.col("ingestion_timestamp").desc_nulls_last(),
        F.col("id").desc_nulls_last()
    )
)

silver_df = (
    silver_df
    .withColumn("row_number", F.row_number().over(window_spec))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

print("Registros después de deduplicar:", silver_df.count())

In [0]:
new_records_count = silver_df.count()

print("Registros a procesar:", new_records_count)

if new_records_count == 0:
    print("No hay registros para cargar en Silver.")

else:
    target = DeltaTable.forName(spark, silver_table)

    (
        target.alias("target")
        .merge(
            silver_df.alias("source"),
            """
            target.nombre_planta = source.nombre_planta
            AND target.nombre_reservorio = source.nombre_reservorio
            """
        )
        .whenMatchedUpdate(set={
            "id": "source.id",
            "region": "source.region",
            "source_file_name": "source.source_file_name",
            "source_file_path": "source.source_file_path",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "load_date": "source.load_date",
            "silver_updated_at": "source.silver_updated_at"
        })
        .whenNotMatchedInsert(values={
            "id": "source.id",
            "region": "source.region",
            "nombre_planta": "source.nombre_planta",
            "nombre_reservorio": "source.nombre_reservorio",
            "source_file_name": "source.source_file_name",
            "source_file_path": "source.source_file_path",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "load_date": "source.load_date",
            "silver_created_at": "source.silver_created_at",
            "silver_updated_at": "source.silver_updated_at"
        })
        .execute()
    )

    print("MERGE de plantas_reservorios ejecutado correctamente.")

In [0]:
silver_count = spark.table(silver_table).count()

assert silver_count > 0, "La tabla silver.plantas_reservorios quedó vacía después del MERGE."

print("Registros actuales en Silver plantas_reservorios:", silver_count)

In [0]:
%sql
SELECT *
FROM observatorio_dev.silver.plantas_reservorios
order BY id

In [0]:
%sql
SELECT
    COUNT(*) AS registros,
    COUNT(DISTINCT CONCAT_WS(
        '||',
        nombre_planta,
        nombre_reservorio
    )) AS relaciones_distintas,
    COUNT(DISTINCT nombre_planta) AS plantas,
    COUNT(DISTINCT nombre_reservorio) AS reservorios
FROM observatorio_dev.silver.plantas_reservorios;